In [80]:
%pip install groq
%pip install -q ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 60.3 MB/s eta 0:00:00


In [116]:
import os
from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)


In [117]:
class Agent:
  def __init__(self, client, system):
    self.cliente = client
    self.system = system
    self.messages = []
    if self.system is not None:
      self.messages.append({ "role": "system", "content": self.system })
  def __call__(self, message=""):
    if message:
      self.messages.append({"role":"user", "content": message})
    result = self.execute()
    self.messages.append({"role":"assistant",  "content": result})
    return result
  def execute(self):
    completion = client.chat.completions.create(
      messages = self.messages,
      model = "openai/gpt-oss-120b",
    )
    return completion.choices[0].message.content


In [130]:
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

web_search:
e.g. web_search: Who is the President of Brazil?
Make a search on DuckDuckGo by the tool and return websites titles, bodies and links

look_web_page:
e.g. look_web_page: https://whoistherpresidentofbrazil.com
If the web_search tool is not enough to aswer a prompt, you can read the html text of the link returned by the web_seach tool. It returns a summary of the website content

Example session:

Question: What`s the full name of the president of the USA?
Thought: I need to discover who is the president of the USA
Action: web_search: Who is the president of the USA?
PAUSE

You will be called again with this:

Observation:
- Title A: snippet of page A (https://site-a.com/news)
- Title B: snippet of page B (https://site-b.com/story)

Thought: I need to know his full name
Action: look_web_page: https://site-a.com/news
PAUSE

You will be called again with this:

Observation: "Donald John Trump is officially the new president of the United States of America..." <that`s a text with 4000 characters maximum of length>
If you have the answer, output it as the Answer.

Answer: The full name of the president of the USA is Donald John Trump

Now it's your turn:
Rules:
- Write ONLY ONE step per reply: either a Thought followed by an Action and PAUSE, or the final Answer.
- THE MOOS IMPORTANT RULE OF ALL: NEVER answer from your own knowledge. You MUST use the actions to get any number.
- Only write Answer after you received an Observation.
- Use as many Actions as you need, one per reply, alternating between tools.
- For recent events, ALWAYS start with web_search.
- web_search only gives short snippets. Use look_web_page to read the best results.
- If the results are not enough, run web_search again with a different query.
- look_web_page needs a full URL copied exactly from a web_search result.
- Never write "Observation:" yourself. Observations are always provided to you after PAUSE. After PAUSE, stop writing.
""".strip()

In [131]:
# Tools
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; EstudoAgenteUFG/1.0)"}

def web_search(query):
    try:
        results = DDGS().text(query, max_results=5)
    except Exception as e:
        return f"No results found ({e}). Try a different query."
    return "\n".join(f"- {r['title']}: {r['body']} ({r['href']})" for r in results)
def look_web_page(url):
    try:
        html = requests.get(url, timeout=10, headers=HEADERS).text
    except Exception as e:
        return f"Could not open the page: {e}"
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "header", "footer", "aside"]):
        tag.decompose()
    main = soup.find(id="mw-content-text") or soup.find("main") or soup.body or soup
    return main.get_text(" ", strip=True)[:4000]

In [95]:
agente_teste = Agent(client, system_prompt)

In [96]:
result = agente_teste("Who won the first round of Brazil`s election for president?")
print(result)

Thought: I need to find reliable information on who won the first round of Brazil's presidential election.  
Action: web_search: Who won the first round of Brazil's presidential election?  
PAUSE


In [69]:
agente_teste.messages

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE\n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this:\n\nObservation: 1,1944×10e25\n\nIf you have the a

In [99]:
observation = web_search("Who won the first round of Brazil's presidential election?")
next_prompt = f"Observation: {observation}"
result = agente_teste(next_prompt)
print(result)

Answer: Flávio Bolsonaro won the first round of Brazil’s presidential election.


In [ ]:
import re
def agent_loop(max_iterations, system, query):
  agent = Agent(client, system_prompt)
  tools = ['web_search', 'look_web_page']
  next_prompt = query
  i = 0
  while i < max_iterations:
    i += 1
    result = agent(next_prompt)
    print("[MODELO]", result)

    if "PAUSE" in result and "Action" in result:
      action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
      chosen_tool = action[0][0]
      arg = action[0][1].strip().strip("\"'")
      if chosen_tool in tools:
        result_tool = eval(f"{chosen_tool}({arg!r})")
        next_prompt = f"Observation: {result_tool}"
      else:
        next_prompt = "Observation: tool not found"

      print("[FERRAMENTA]", next_prompt[:500])
      continue
    if "Answer" in result:
      break
    next_prompt = ("You did not write an Action. Reply with a Thought, "
                   "then 'Action: tool: input', then PAUSE.")

In [135]:
agent_loop(max_iterations=15, system=system_prompt,query="Who were the 5 main candidates who ran in the 2026 Brazil`s presidential election?")

PAUSE
PAUSE
Thought: I need to find information about the five main candidates who ran in Brazil's 2026 presidential election.  
Action: web_search: 2026 Brazil presidential election main candidates  
PAUSE
Observation: - 2026 Brazilian general election - Wikipedia: General elections were held in Brazil on 4 October 2026 to elect the president, vice president, members of the National Congress, the governors, vice governors and legislative assemblies of all federative units, and the district council of Fernando de Noronha. (https://en.wikipedia.org/wiki/2026_Brazilian_general_election)
- Brazil: Meet the Candidates 2026 - americasquarterly.org: Aug 3, 2026 · Brazil’s presidential race has been dominated by two highly polarized candidates: President Luiz Inácio Lula da Silva, who is seeking a fourth term, and Senator Flávio Bolsonaro, a son of former President Jair Bolsonaro. (https://www.americasquarterly.org/article/brazil-meet-the-candidates-2026/)
- Brazil’s 2026 Presidential Candida